# 04 Recommender Models


In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path('..')/'src'))


## Objectives

Build and compare **recommendation models**:

1. **Popularity baseline**: rank by `num_subscribers` or `profit`.
2. **Content-based TF-IDF**: cosine similarity on `course_title`.
3. **Hybrid re-ranking**: combine similarity with predicted profitability.

### Offline evaluation (weak labels)
Because the dataset has no user-course interactions, we use `subject` as a weak relevance label:
- A recommended course is considered *relevant* if it shares the same `subject` as the seed.

We report **Precision@K** (and optionally NDCG@K).


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import joblib
from scipy import sparse

DATA_PATH = Path('..')/'data'/'processed'/'courses_clean.parquet'
df = pd.read_parquet(DATA_PATH)
df.shape


(3677, 19)

In [3]:
# Load profitability model from previous notebook
MODEL_PATH = Path('..')/'models'/'profit_model.joblib'
model_bundle = joblib.load(MODEL_PATH)
pipe = model_bundle['pipeline']

pipe


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sp

### A) Popularity baseline

This is a non-personalized recommender: for a subject filter, recommend top-K courses by `profit` (or `num_subscribers`).


In [4]:
def popularity_recs(df, subject=None, top_k=10, sort_col='profit'):
    x=df.copy()
    if subject is not None:
        x=x[x['subject']==subject]
    return x.sort_values(sort_col, ascending=False).head(top_k)

popularity_recs(df, subject='Web Development', top_k=5)[['course_title','profit','num_subscribers','price']]


,course_title,profit,num_subscribers,price
3229,The Web Developer Bootcamp,24316800,121584,200
3231,The Complete Web Developer Course 2.0,22902400,114512,200
3203,Angular 4 (formerly Angular 2) - The Complete ...,14018770,73783,190
3246,JavaScript: Understanding the Weird Parts,13932100,79612,175
3250,Learn and Understand NodeJS,11350560,58208,195


### B) TF-IDF item-to-item recommender


In [5]:
from course_rec.recommender import build_tfidf_recommender, recommend_similar_courses

rec = build_tfidf_recommender(df, text_col='course_title', max_features=20000)

# Example: pick a seed course
seed_id = int(df.iloc[0]['course_id'])
seed_title = df.iloc[0]['course_title']
seed_id, seed_title


(1070968, 'Ultimate Investment Banking Course')

In [6]:
recs = recommend_similar_courses(df, rec, seed_course_id=seed_id, top_k=10)
recs[['course_title','subject','similarity','profit','num_subscribers']].head(10)


,course_title,subject,similarity,profit,num_subscribers
39,The Complete Investment Banking Course 2017,Business Finance,0.765072,1672125,8575
419,The Investment Banking Recruitment Series,Business Finance,0.642187,680,17
242,Advanced Accounting for Investment Banking,Business Finance,0.542051,63000,1260
229,Investment Banking: How to Land a Job on Wall ...,Business Finance,0.418505,91350,1218
139,"Intro to Investment Banking, M&A, IPO, Modelin...",Business Finance,0.415100,115320,1922
724,Investment Banking Operations : Securities Tra...,Business Finance,0.397720,8010,267
420,Business Banking 101,Business Finance,0.281579,3300,132
3473,The Ultimate jQuery Course,Web Development,0.251296,21960,1098
3009,PHP with PDO - ULTIMATE Crash Course,Web Development,0.246040,62400,1248
452,Complete Claritas Investment Certificate,Business Finance,0.226893,19400,388


### C) Hybrid re-ranking: Similarity + Predicted Profit

We score candidate courses as:

\[ score = \alpha \cdot \text{sim} + (1-\alpha) \cdot \widehat{profit} \]

where both terms are min-max normalized within the candidate set.


In [7]:
from course_rec.recommender import hybrid_rerank

# Predict profitability for candidates
candidates = recs.copy()

# For prediction, we need the same structured columns as in training
num_cols = model_bundle['numeric_cols']
cat_cols = model_bundle['categorical_cols']

X_cand = candidates[num_cols + cat_cols]
pred_log = pipe.predict(X_cand)
candidates['predicted_profit'] = np.expm1(pred_log)

hyb = hybrid_rerank(candidates, alpha=0.65, top_k=10)
hyb[['course_title','subject','similarity','predicted_profit','hybrid_score']].head(10)


,course_title,subject,similarity,predicted_profit,hybrid_score
39,The Complete Investment Banking Course 2017,Business Finance,0.765072,1.673150e+06,1.000000
419,The Investment Banking Recruitment Series,Business Finance,0.642187,6.855060e+02,0.501583
242,Advanced Accounting for Investment Banking,Business Finance,0.542051,6.200565e+04,0.393474
139,"Intro to Investment Banking, M&A, IPO, Modelin...",Business Finance,0.415100,1.133465e+05,0.250889
229,Investment Banking: How to Land a Job on Wall ...,Business Finance,0.418505,9.152003e+04,0.250434
724,Investment Banking Operations : Securities Tra...,Business Finance,0.397720,8.016516e+03,0.207855
420,Business Banking 101,Business Finance,0.281579,3.348151e+03,0.066606
3009,PHP with PDO - ULTIMATE Crash Course,Web Development,0.246040,6.183103e+04,0.035921
3473,The Ultimate jQuery Course,Web Development,0.251296,2.245640e+04,0.034029
452,Complete Claritas Investment Certificate,Business Finance,0.226893,1.930910e+04,0.003897


## Evaluation: Precision@K (subject as weak label)

For each course as a seed:
- recommend top-K similar courses
- compute precision@K = (# recommendations with same subject) / K

We report the mean precision@K across all seeds.


In [8]:
def precision_at_k_subject(df, rec, k=10, sample_n=500, seed=42):
    rng = np.random.default_rng(seed)
    seeds = df['course_id'].to_numpy()
    if sample_n is not None and sample_n < len(seeds):
        seeds = rng.choice(seeds, size=sample_n, replace=False)

    subj = df.set_index('course_id')['subject'].to_dict()
    precs=[]

    for cid in seeds:
        # recommend k (content)
        r = recommend_similar_courses(df, rec, seed_course_id=int(cid), top_k=k)
        rel = (r['subject'] == subj[int(cid)]).mean() if len(r)>0 else 0.0
        precs.append(rel)

    return float(np.mean(precs)), float(np.std(precs))

mean_p10, std_p10 = precision_at_k_subject(df, rec, k=10, sample_n=500)
(mean_p10, std_p10)


(0.8208000000000001, 0.23547263110603747)

In [9]:
# Hybrid evaluation: rerank the TF-IDF candidate list and re-measure precision@K

def precision_at_k_hybrid(df, rec, pipe, model_bundle, k=10, alpha=0.65, sample_n=500, seed=42):
    rng = np.random.default_rng(seed)
    seeds = df['course_id'].to_numpy()
    if sample_n is not None and sample_n < len(seeds):
        seeds = rng.choice(seeds, size=sample_n, replace=False)

    subj = df.set_index('course_id')['subject'].to_dict()
    precs=[]

    num_cols = model_bundle['numeric_cols']
    cat_cols = model_bundle['categorical_cols']

    for cid in seeds:
        cands = recommend_similar_courses(df, rec, seed_course_id=int(cid), top_k=50)  # broader pool
        X = cands[num_cols + cat_cols]
        pred_log = pipe.predict(X)
        cands['predicted_profit'] = np.expm1(pred_log)

        top = hybrid_rerank(cands, alpha=alpha, top_k=k)
        rel = (top['subject'] == subj[int(cid)]).mean() if len(top)>0 else 0.0
        precs.append(rel)

    return float(np.mean(precs)), float(np.std(precs))

mean_h10, std_h10 = precision_at_k_hybrid(df, rec, pipe, model_bundle, k=10, alpha=0.65, sample_n=500)
(mean_h10, std_h10)


(0.8121999999999999, 0.23847674939079494)

## Save artifacts for Streamlit

We persist:
- `tfidf_vectorizer.joblib`
- `tfidf_matrix.npz`
- `course_index.parquet`


In [10]:
from pathlib import Path
import joblib
from scipy import sparse

models_dir = Path('..')/'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(rec.vectorizer, models_dir/'tfidf_vectorizer.joblib')
sparse.save_npz(models_dir/'tfidf_matrix.npz', rec.tfidf_matrix)

# Minimal index for the app
idx_cols = ['course_id','course_title','url','subject','level','is_paid','price','num_subscribers','num_reviews','num_lectures','content_duration_hours','year','month','day','profit']
df[idx_cols].to_parquet(models_dir/'course_index.parquet', index=False)

(models_dir/'tfidf_vectorizer.joblib', models_dir/'tfidf_matrix.npz', models_dir/'course_index.parquet')


(WindowsPath('../models/tfidf_vectorizer.joblib'),
 WindowsPath('../models/tfidf_matrix.npz'),
 WindowsPath('../models/course_index.parquet'))